# Домашнее задание: доменная адаптация LLM

В рамках этого ноутбука мы попробуем решить довольно популярную сейчас в российских IT-компаниях задачу, правда, в более маленьком масштабе. Мы затюним китайскую модель под русский язык.
Что охватим:
1. Замеры бенчмарков сторонними библиотеками
2. Токенизация и построение датасета
3. Обучение LoRA модели
4. Инференс

# 0. Important imports

Для начала импортируем нужные штуки. Будем использовать библиотеку от https://unsloth.ai. Они написали очень оптимальный код для дообучения LLM под свои задачи.

In [ ]:
%%capture
import os
if "COLAB_" not in "".join(os.environ.keys()):
    %pip install unsloth
else:
    # Do this only in Colab notebooks! Otherwise use pip install unsloth
    !pip install --no-deps bitsandbytes accelerate xformers==0.0.29.post3 peft trl triton cut_cross_entropy unsloth_zoo
    !pip install sentencepiece protobuf "datasets>=3.4.1" huggingface_hub hf_transfer
    !pip install --no-deps unsloth

# 1. Base model — 10 баллов

В качестве базовой модели возьмем Qwen2.5-1.5B — это достаточно маленькая модель, она спокойно и быстро обучится в Google Colab. Давайте скачаем ее локально в виде директории, чтобы иметь к ней быстрый доступ. Будем скачивать ее из репозитория https://huggingface.co/unsloth/Qwen2.5-1.5B. Убедитесь, что у вас установлен `git lfs`.

In [ ]:
# https://github.com/git-lfs/git-lfs/blob/main/INSTALLING.md
!git clone https://huggingface.co/unsloth/Qwen2.5-1.5B

### FastLanguageModel — 3 балла
Обернем нашу модель в класс `FastLanguageModel` из unsloth, документацию можно найти [тут](https://docs.unsloth.ai/get-started/fine-tuning-guide#id-2.-choose-the-right-model--method). max_seq_length установим в 2048 и загрузим модель в 4 битах для ускорения обучения.

In [ ]:
from unsloth import FastLanguageModel
import torch
# ---- Ваш код здесь ----

# Видимо, тут параметры зададим

# Максимальная длина последовательности в токенах.
max_seq_length = 2048

# Unsloth умный, автоматически выберет тип вычислений под доступную GPU.
dtype = None

# Загружаем веса в 4-битном формате. Кто-то говорит "ускорение", а я говорю "экономия видеопамяти".
load_in_4bit = True

# ---- Конец кода ----


# ---- Ваш код здесь ----
import sys
import platform
from importlib.metadata import version, PackageNotFoundError

print("Операционная система:", platform.platform())
print("Версия Python:", sys.version)
print("Путь к Python:", sys.executable)

# Замучался удовлетворять зависимости руками. Пусть это останется здесь.
for package in ["torch", "unsloth", "unsloth_zoo", "transformers", "vllm"]:
    try:
        print(f"{package}: {version(package)}")
    except PackageNotFoundError:
        print(f"{package}: не установлен")
# ---- Конец кода ----

model, tokenizer = FastLanguageModel.from_pretrained(
    # ---- Ваш код здесь ----
    model_name="./Qwen2.5-1.5B",  # Путь к скачанной модели. 3ГБ получилось.
    max_seq_length=max_seq_length,
    dtype=dtype,
    load_in_4bit=load_in_4bit,
    device_map={"": "cuda:0"},  # Всю модель размещаем на первой CUDA-видеокарте.
    # ---- Конец кода ----
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


W1015 15:55:55.225000 40516 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.


🦥 Unsloth Zoo will now patch everything to make training faster!
Операционная система: Windows-11-10.0.26300-SP0
Версия Python: 3.13.9 (tags/v3.13.9:8183fa5, Oct 14 2025, 14:09:13) [MSC v.1944 64 bit (AMD64)]
Путь к Python: c:\igori\pytorch\.venv\Scripts\python.exe
torch: 2.9.1+cu130
unsloth: 2026.3.18
unsloth_zoo: 2026.3.7
transformers: 4.57.1
vllm: не установлен
==((====))==  Unsloth 2026.3.18: Fast Qwen2 patching. Transformers: 4.57.1.
   \\   /|    NVIDIA GeForce RTX 5080. Num GPUs = 1. Max memory: 15.92 GB. Platform: Windows.
O^O/ \_/ \    Torch: 2.9.1+cu130. CUDA: 12.0. CUDA Toolkit: 13.0. Triton: 3.5.1
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
./Qwen2.5-1.5B does not have a padding token! Will use pad_token = <|PAD_TOKEN|>.


### Генерируем ответ базовой моделью — 4 балла

Теперь давайте попробуем оценить возможности нашей модели прямо из коробки. Для этого напишем функцию `generate_answer`, которая будет принимать на вход промпт и возвращать нам ответ. Несмотря на то, что наша модель является базовой, то есть не обученной под диалоговый формат, мы все равно предполагаем, что диалог она понимает. Оформите функцию так, чтобы можно было добавить опционально шаблон диалога и сравнить результаты с ним и без него:

In [28]:
from transformers import AutoTokenizer
FastLanguageModel.for_inference(model)
qwen_tokenizer = AutoTokenizer.from_pretrained("Qwen/Qwen2.5-7B")

def generate_answer(prompt, apply_chat_template=False):
    # ---- Ваш код здесь ----
    if apply_chat_template:
        # Строка — один запрос; список — готовая история разговора. Это на будущее, для диалога.
        if isinstance(prompt, str):
            messages = [{"role": "user", "content": prompt}]
        else:
            messages = prompt

        prompt = qwen_tokenizer.apply_chat_template(
            messages,
            tokenize=False,             # Пока получаем текст, а не номера токенов. Токенайзер-то этот от другой модели, с него только шаблон.
            add_generation_prompt=True, # Добавляем начало реплики ассистента.
        )
        #print(prompt)

    # тут промпт уже с опциональным шаблоном, таким же как при тестах после обучения (ранее: от старшей модели)


    inputs = tokenizer(
        prompt,
        return_tensors="pt",  # Возвращаем тензоры PyTorch, уже в токенах нашей модели.
        # Шаблон уже содержит специальные токены — не добавляем их повторно.
        add_special_tokens=not apply_chat_template,
    ).to(model.device)  # Входные данные должны находиться на той же GPU, что и модель.

    # Отключаем вычисление градиентов: сейчас мы генерируем, а не обучаем.
    with torch.inference_mode():
        outputs = model.generate(
            **inputs,              # Передаём input_ids и attention_mask.
            max_new_tokens=256,    # Ограничиваем длину только нового ответа.
            do_sample=False,       # Для начала по самому очевидному пути.
            pad_token_id=tokenizer.eos_token_id,
        )

    # generate возвращает исходный промпт вместе с продолжением.
    # Отрезаем промпт, чтобы функция возвращала только ответ.
    prompt_length = inputs["input_ids"].shape[1]
    answer_tokens = outputs[0, prompt_length:]

    return tokenizer.decode(answer_tokens, skip_special_tokens=True).strip()
    # ---- Конец кода ----

# надо посомтреть, как там шаблон влияет

prompt = "Расскажи мне про Булгакова"

print("Без шаблона:")
print(generate_answer(prompt))

print("\nС шаблоном диалога:")
print(generate_answer(prompt, apply_chat_template=True))

# Да, полтора миллиона обезъян за печатными машинками...

#print(generate_answer("Расскажи мне про Булгакова"))

Без шаблона:
Булгаков — один из самых известных и влиятельных писателей XX века, чьи произведения включены в школьный курс по литературе. Его творчество, особенно роман «Отцы и дети», имеет огромное влияние на современное искусство и литературу. Вот несколько основных сюжетов и персонажей, которые можно рассказать о Булгакове:

1. **«Отцы и дети»**: Это роман, который рассказывает о жизни и переживаниях семьи Булгаковых, включая отца, матери, деда и дедушки. Роман является одним из самых известных произведений Булгакова и имеет большое влияние на современное искусство и литературу.

2. **«Война и мир»**: Это роман, который рассказывает о жизни в СССР в периоде Второй мировой войны. Роман является одним из самых известных произведений Булгакова и имеет большое влияние на современное искусство и литературу.

3. **«Человек

С шаблоном диалога:
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
rame
ra

In [16]:
# ---- Ваш код здесь ----

# добавление шаблона приводит модель в полный неадекват. Попробуем проверить причину

# вот тут похоже собака порылась, кривые маркеры из старшей модели не нравятся. Хотя родной токенайзер их вполне понимает.
for marker in ["<|im_start|>", "<|im_end|>"]:
    token_ids = tokenizer.encode(marker, add_special_tokens=False)
    print(f"{marker}: {token_ids}")

#question = "Расскажи мне про Булгакова"
question = "Tell me about Lenin"
#question = "天空是什么颜色的"

original_template = qwen_tokenizer.apply_chat_template(
    [{"role": "user", "content": question}],
    tokenize=False,
    add_generation_prompt=True,
)

variants = {
    "Без шаблона": question,

    "Исходный шаблон": original_template,

    # Формируем вручную: шаблон Qwen сам добавляет system,
    # даже если мы передаём только сообщение пользователя.
    "Без system": (
        f"<|im_start|>user\n{question}<|im_end|>\n"
        "<|im_start|>assistant\n"
    ),

    "System на русском": qwen_tokenizer.apply_chat_template(
        [
            {"role": "system", "content": "Ты полезный помощник."},
            {"role": "user", "content": question},
        ],
        tokenize=False,
        add_generation_prompt=True,
    ),

    # Сохраняем роли, но убираем специальные маркеры.
    "Роли без специальных маркеров": (
        f"user\n{question}\nassistant\n"
    ),

    "Обычный текстовый диалог": (
        f"Вопрос: {question}\nОтвет:"
    ),
}

for name, text in variants.items():
    inputs = tokenizer(
        text,
        add_special_tokens=False,
        return_tensors="pt",
    ).to(model.device)

    print(f"\n--- {name} ---")
    # repr показывает переводы строк и точный конец промпта.
    print("Вход:", repr(text))

    with torch.inference_mode():
        outputs = model.generate(
            **inputs,
            max_new_tokens=64,
            do_sample=False,  # Во всех вариантах одинаковая генерация.
            pad_token_id=tokenizer.eos_token_id,
        )

    new_tokens = outputs[0, inputs["input_ids"].shape[1]:]
    print(
        "Продолжение:",
        repr(tokenizer.decode(new_tokens, skip_special_tokens=False)),
    )

# Небо какого цвета?
# 
print(generate_answer("天空是什么颜色的？", apply_chat_template=True))

# ---- Конец кода ----

<|im_start|>: [151644]
<|im_end|>: [151645]

--- Без шаблона ---
Вход: 'Tell me about Lenin'
Продолжение: "'s political philosophy.\nLenin's political philosophy was based on the idea of a socialist revolution and the establishment of a communist society. He believed that the working class should take control of the means of production and use it to serve the needs of the people. He also believed in the need for a strong central government to ensure"

--- Исходный шаблон ---
Вход: '<|im_start|>system\nYou are a helpful assistant.<|im_end|>\n<|im_start|>user\nTell me about Lenin<|im_end|>\n<|im_start|>assistant\n'
Продолжение: 'Tell me about Lenin实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n实事求\n'

--- Без system ---
Вход: '<|im_start|>user\nTell me about Lenin<|im_end|>\n<|im_start|>assistant\n'
Продолжение: 'Tell me about LeninốAssistant\nTell me about LeninốAssistant\nTell me about Len

### Оцениваем возможности и собираем корзинку — 3 балла

Запишите свои выводы в ячейку с markdown ниже. Меняется ли поведение модели при добавлении диалога? Попробуйте промпты на русском, английском и (на всякий случай) китайском.

**Выводы:**
\# Ага, я как раз в предыдущей части это посмотрел и изумился. Промпты на русском и английском нормально, а маркеры <|im_start|>, <|im_end|> в шаблоне всё портят.
НО: если дать вопрос на _китайском_, и шаблон не трогать - оставить один в один из токенизатора, то отвечает внезапно связно.

**Видимо, в привычной среде модель ориентируется лучше.**

_ИО_

Придумайте "корзинку" из 10 диалоговых промптов на русском языке. Выведите ответы модели на этой корзинке, по ней будем оценивать результат обучения. Можете придумать сами или взять интересные примеры отсюда: https://llmarena.ru/ -> Arena Explorer. Учтите, что проверять стоит скорее диалоговость, а не знания модели.

In [30]:
# ---- Ваш код здесь ----
prompts = [
    # 1. Простой ответ на приветствие.
    "Привет! Как дела?",

    # 2. Реакция на знакомство.
    "Меня зовут Игорь. Давай познакомимся!",

    # 3. Завершение разговора.
    "Спасибо за помощь! До завтра.",

    # 4. Уместная поддержка без сложных рекомендаций.
    "Сегодня был тяжёлый день, и я очень устал. Скажи мне что-нибудь доброе.",

    # 5. Простая бытовая рекомендация.
    "Мне скучно дома. Предложи одно занятие.",

    # 6. Выполнение короткой просьбы: готовый текст вместо рассуждений.
    "Напиши короткое поздравление другу с днём рождения.",

    # 7. Простое изменение тона.
    "Как вежливо сказать: «Дай мне ручку»?",

    # 8. Использование явно указанной информации.
    "Моего кота зовут Барсик. Как зовут моего кота?",

    # 9. Один понятный запрет.
    "Предложи имя для кошки. Только не Мурка — такая кошка у нас уже есть.",

    # 10. контекст диалога и уточнение, в формате диалога
    [
        {"role": "user", "content": "Посоветуй, что выпить."},
        {"role": "assistant", "content": "Чай или кофе?"},
        {"role": "user", "content": "Кофе не хочу. Что тогда выбрать?"},
    ],
]

# Сохраняем выбранный режим для повторного прогона после обучения.

# ---- Ваш код здесь ----
import json
from pathlib import Path

result_path = Path("baseline_two_modes.json")

if result_path.exists():
    raise FileExistsError(
        f"Файл {result_path} уже существует. "
        "Для нового замера укажи другое имя файла."
    )

base_answers = []

for number, prompt in enumerate(prompts, start=1):
    messages = (
        [{"role": "user", "content": prompt}]
        if isinstance(prompt, str)
        else prompt
    )

    if isinstance(prompt, str):
        # Одиночный вопрос передаём без изменений.
        plain_prompt = prompt
    else:
        # Историю сохраняем, но записываем обычным текстом без ChatML-маркеров.
        role_names = {
            "system": "Инструкция",
            "user": "Пользователь",
            "assistant": "Помощник",
        }
        plain_prompt = "\n".join(
            f"{role_names[message['role']]}: {message['content']}"
            for message in messages
        )
        plain_prompt += "\nПомощник:"

    # Сохраняем точную строку, которую шаблон передаёт токенизатору модели.
    chat_prompt = qwen_tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True,
    )

    answer_plain = generate_answer(
        plain_prompt,
        apply_chat_template=False,
    )
    answer_chat = generate_answer(
        messages,
        apply_chat_template=True,
    )

    base_answers.append({
        "number": number,
        "messages": messages,
        "without_chat_template": {
            "input_text": plain_prompt,
            "answer": answer_plain,
            "score": None,
        },
        "with_chat_template": {
            "input_text": chat_prompt,
            "answer": answer_chat,
            "score": None,
        },
    })

    print(f"\n{'=' * 60}\nЗадание {number}")
    for message in messages:
        print(f"{message['role']}: {message['content']}")

    print(f"\nБез шаблона:\n{answer_plain}")
    print(f"\nС шаблоном Qwen:\n{answer_chat}")

baseline_data = {
    "model": "./Qwen2.5-1.5B",
    "stage": "before_training",
    "template_source": "Qwen/Qwen2.5-7B",
    "chat_template": qwen_tokenizer.chat_template,
    "add_generation_prompt": True,
    # Параметры текущей функции generate_answer одинаковы для обоих режимов.
    "generation": {
        "do_sample": False,
        "max_new_tokens": 256,
        "pad_token_id": tokenizer.eos_token_id,
    },
    "scoring": {
        "0": "Нет осмысленного ответа по запросу.",
        "1": "Ответ по теме, но просьба выполнена частично.",
        "2": "Уместный ответ с выполнением просьбы.",
    },
    "results": base_answers,
}

result_path.write_text(
    json.dumps(baseline_data, ensure_ascii=False, indent=2),
    encoding="utf-8",
)

print(f"\nОба варианта сохранены: {result_path.resolve()}")
# ---- Конец кода ----


Задание 1
user: Привет! Как дела?

Без шаблона:
Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти 

# 2. Бенчмарки — 10 баллов

Давайте замерим нашу модель с помощью бенчмарка LM Evaluation Harness — документацию можно найти [тут](https://github.com/EleutherAI/lm-evaluation-harness), установить его можно по инструкции из репозитория. Это нужно будет, чтобы убедиться, что мы не просадили качество нашим обучением. Запускать бенчмарк будем из командной строки.

In [ ]:
!git clone --depth 1 https://github.com/EleutherAI/lm-evaluation-harness
!cd lm-evaluation-harness && pip install -e .

Если будет падать с ошибкой `HTTP Error 429`, то придется залогиниться в HF. Это особенность колаба

In [ ]:
# from huggingface_hub import notebook_login
# notebook_login()

Напишите bash скрипт для прогона бенчмарков:

In [22]:
%%writefile run_lmeh.sh
#!/usr/bin/env bash
# ---- Ваш код здесь ----
set -euo pipefail

# Используем интерпретатор, переданный из ядра ноутбука.
# При запуске вручную — python из активированной среды.
PYTHON="${PYTHON:-python}"

# Единая кодировка вывода и немедленная передача сообщений.
export PYTHONIOENCODING=utf-8
export PYTHONUNBUFFERED=1

"$PYTHON" -u -m lm_eval run \
    --model hf \
    --model_args "pretrained=./Qwen2.5-1.5B,dtype=float16,load_in_4bit=True,bnb_4bit_quant_type=nf4,bnb_4bit_compute_dtype=float16,bnb_4bit_use_double_quant=True" \
    --tasks truthfulqa_ru_mc1,copa,wsc273 \
    --num_fewshot 0 \
    --device cuda:0 \
    --batch_size 8 \
    --seed 42 \
    --output_path ./results/base_model

# ---- Конец кода ----

Overwriting run_lmeh.sh


In [25]:
# ---- Ваш код здесь ----
import os
import sys
import subprocess
from pathlib import Path

script_path = Path("run_lmeh.sh")

# башу не нравятся виндовые концы строк
script_path.write_bytes(
    script_path.read_bytes().replace(b"\r\n", b"\n")
)

env = os.environ.copy()

# Передаём путь через окружение, без вложенного экранирования кавычек.
# Прямые слеши удобны для запуска Windows Python из Git Bash.
env["PYTHON"] = Path(sys.executable).as_posix()
# Обновляем индикаторы прогресса не чаще одного раза в 5 секунд.
env["TQDM_MININTERVAL"] = "5"

with open("run_lmeh.log", "w", encoding="utf-8") as log_file:
    with subprocess.Popen(
        # Явно используем Git Bash, который запускает Windows Python из нашей .venv. r-строка не трогает бэкслэши.
        [r"C:\Program Files\Git\bin\bash.exe", str(script_path)],        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # Ошибки и обычные сообщения читаем вместе.
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()

        return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Harness завершился с кодом {return_code}. "
        "Подробности — в выводе выше и в run_lmeh.log."
    )
# ---- Конец кода ----

C:\igori\pytorch\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
2026-10-15:17:50:24 INFO     [_cli.run:413] Selected Tasks: ['truthfulqa_ru_mc1', 'copa', 'wsc273']
2026-10-15:17:50:24 INFO     [evaluator:214] Setting random seed to 42 | Setting numpy seed to 42 | Setting torch manual seed to 42 | Setting fewshot manual seed to 42
2026-10-15:17:50:24 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': './Qwen2.5-1.5B', 'dtype': 'float16', 'load_in_4bit': True, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_use_double_quant': True}
2026-10-15:17:50:26 INFO     [models.huggingface:289] Using device 'cuda:0'
2026-10-15:17:50:26 INFO     [models.huggingface:582] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
W1015 17:50:26.708000 42536 .

В качестве бенчмарков выберете `truthfulqa_ru_mc1` и еще 2 любых на ваше усмотрение — главное, чтобы они быстро пробежали (не более 10 минут). В ответ напишите 3 строчки с результатами бенчмарков.

**Результаты замеров:** 

Выберем copa — выбор причины или следствия, и wsc273 — определение, к кому относится местоимение. Это небольшие англоязычные тесты, которые дополнят русский TruthfulQA.

| Бенчмарк | Accuracy | Стандартная ошибка |
|:---|---:|---:|
| truthfulqa_ru_mc1 | 28.93% | 1.62% |
| copa | 81.00% | 3.94% |
| wsc273 | 76.19% | 2.58% |

# 3. Готовим данные — 5 баллов

Довольно часто возникает задача адаптации базовой модели под русский язык. В рамках этого ноутбука хочется обучиться на датасете с русскоязычными инструкциями.
Для дообучения модели будем использовать датасет `Vikhrmodels/GrandMaster-PRO-MAX`. Было решено остановиться именно на нем, так как он самый свежий и модели, обученные на этом датасете, показывают хорошее качество.

Однако на HuggingFace много и других инструктивных датасетов под русский язык, например тут: https://huggingface.co/collections/IlyaGusev/

Очень часто вы будете сталкиваться с разными шаблонами диалога, как-то модифицировать его или даже готовить собственный. Давайте в рамках подготовки датасета для обучения создадим несложный шаблон вида:

```text
<|im_start|>user
Hi there!<|im_end|>
<|im_start|>assistant
Nice to meet you!<|im_end|>
<|im_start|>user
Can I ask a question?<|im_end|>
```

Про шаблоны можно почитать тут:
https://huggingface.co/docs/transformers/en/chat_templating

In [31]:
def formatting_func(example):
    # ---- Ваш код здесь ----
    parts = []

    for message in example["conversation"]:
        # Каждая реплика содержит роль, текст и маркер завершения.
        # Перевод строки после <|im_end|> нужен и у последней реплики.
        parts.append(
            f"<|im_start|>{message['role']}\n"
            f"{message['content']}<|im_end|>\n"
        )

    # map добавит полученную строку в колонку text.
    return {"text": "".join(parts)}
    # ---- Конец кода ----

In [32]:
chat = { "conversation": [
  {"role": "user", "content": "Hello!"},
  {"role": "assistant", "content": "How can I help you today?"},
  {"role": "user", "content": "I dont know!"},
]}

assert formatting_func(chat)["text"] == '<|im_start|>user\nHello!<|im_end|>\n<|im_start|>assistant\nHow can I help you today?<|im_end|>\n<|im_start|>user\nI dont know!<|im_end|>\n'

Шаблонизируем наш датасет:

In [33]:
from datasets import load_dataset
vikhr_dataset = load_dataset("Vikhrmodels/GrandMaster-PRO-MAX", split = "train")
vikhr_dataset = vikhr_dataset.map(formatting_func, batched = False,)

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00002.parquet:   0%|          | 0.00/139M [00:00<?, ?B/s]

data/train-00001-of-00002.parquet:   0%|          | 0.00/122M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/5.85M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/151822 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/3291 [00:00<?, ? examples/s]

Map:   0%|          | 0/151822 [00:00<?, ? examples/s]

# 4. Обучаем нашу модель — 10 баллов

### Конфигурация LoRA — 5 баллов

Мы будем проводить обучение на инструктивный датасет с помощью библиотеки `TRL SFT`, ее документация [тут](https://huggingface.co/docs/trl/sft_trainer). Ваша задача — подобрать конфигурацию для обучения. Обучение должно уложиться в 1 час на GPU T4. В качестве `target_modules` возьмите все матрицы из подблока attention и mlp, их должно быть 7. Остальные гиперпараметры можно варьировать.

In [34]:
model = FastLanguageModel.get_peft_model(
    model,
    # ---- Ваш код здесь ----
    # всё подбираю под установленную RTX5080
    
    r=16,  # Ранг адаптеров: умеренная ёмкость для первого обучения.

    # Четыре проекции внимания и три проекции MLP — всего семь.
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],

    lora_alpha=32,  # Масштаб LoRA: при обычной LoRA множитель равен alpha / r.
    lora_dropout=0,
    bias="none",   # Смещения базовой модели не обучаем.

    # Экономим память на активациях за счёт повторных вычислений.
    use_gradient_checkpointing="unsloth",
    random_state=42,    
    # ---- Конец кода ----
)

Unsloth 2026.3.18 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


### Обучение модели — 5 баллов

In [37]:
from trl import SFTConfig, SFTTrainer
#trainer = SFTTrainer(
#    model = model,
#    tokenizer = tokenizer,
#    train_dataset = vikhr_dataset,
#    dataset_text_field = "text",
#    max_seq_length = max_seq_length,
#    dataset_num_proc = 2,
#    packing = False, # Can make training 5x faster for short sequences.
    # ---- Ваш код здесь ----
    # ---- Конец кода ----
#)

# Давно уже требуют наличия отдельного SFTConfig, сделаем с ним

# ---- Ваш код здесь ----

# Убираем оставшиеся флаги инференса только там, где они реально хранятся.
# Проверка через vars не обращается к атрибутам вложенной модели.
for module in model.modules():
    if "_flag_for_generation" in vars(module):
        delattr(module, "_flag_for_generation")

# Теперь штатно включаем режим обучения.
FastLanguageModel.for_training(model)
model.config.use_cache = False

# Берём случайную воспроизводимую подвыборку.
# Это ограничит и обучение, и предварительную токенизацию большого датасета.
train_dataset = (
    vikhr_dataset
    .shuffle(seed=42)
    .select(range(min(8000, len(vikhr_dataset))))
    .select_columns(["text"])
)

training_args = SFTConfig(
    output_dir="qwen_lora_checkpoints",

    # Настройки подготовки текста.
    dataset_text_field="text",
    max_length=max_seq_length,  # 2048; более длинные примеры будут обрезаны.
    dataset_num_proc=None,  # Токенизация в текущем процессе, без multiprocessing.
    packing=False,

    # В нашем формате конец реплики обозначается именно этим токеном.
    eos_token="<|im_end|>",

    # За один проход обрабатываем 4 примера.
    # Обновляем веса после накопления градиентов от 4 проходов.
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,

    # Ограничиваем число обновлений весов, а не число эпох.
    max_steps=500,
    learning_rate=1e-4,
    warmup_steps=25,            # Первые 25 шагов плавно увеличиваем learning rate.
    lr_scheduler_type="linear",
    weight_decay=0.01,
    max_grad_norm=1.0,          # Ограничиваем слишком большие градиенты.

    # RTX 5080 поддерживает BF16; поддержка подтверждена и логом Unsloth.
    bf16=True,
    fp16=False,

    # Обычный AdamW достаточен: обучаем только небольшие LoRA-адаптеры.
    optim="adamw_torch",
    gradient_checkpointing=True,

    # У нас готовые строки text: обучаем предсказанию всей последовательности.
    completion_only_loss=False,
    assistant_only_loss=False,

    logging_steps=10,
    save_strategy="steps",
    save_steps=100,
    save_total_limit=2,         # Храним только два последних чекпоинта.
    report_to="none",           # Без подключения внешних сервисов.
    dataloader_num_workers=0,   # Без дополнительных процессов загрузки в Windows.
    seed=42,
)

trainer = SFTTrainer(
    model=model,
    processing_class=tokenizer,
    train_dataset=train_dataset,
    args=training_args,
)
# ---- Конец кода ----

Unsloth: Tokenizing ["text"]:   0%|          | 0/8000 [00:00<?, ? examples/s]

🦥 Unsloth: Padding-free auto-enabled, enabling faster training.


In [38]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA GeForce RTX 5080. Max memory = 15.92 GB.
1.598 GB of memory reserved.


Запускаем обучение:

In [39]:
trainer_stats = trainer.train()

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 8,000 | Num Epochs = 1 | Total steps = 500
O^O/ \_/ \    Batch size per device = 4 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (4 x 4 x 1) = 16
 "-____-"     Trainable parameters = 18,464,768 of 1,562,179,072 (1.18% trained)


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
10,1.418200
20,1.363100
30,1.284100
40,1.333900
50,1.260600
60,1.218600
70,1.233600
80,1.232600
90,1.204300
100,1.187400


На всякий случай сохраним прогресс:

In [40]:
model.save_pretrained("lora_model")
tokenizer.save_pretrained("lora_model")

('lora_model\\tokenizer_config.json',
 'lora_model\\special_tokens_map.json',
 'lora_model\\vocab.json',
 'lora_model\\merges.txt',
 'lora_model\\added_tokens.json',
 'lora_model\\tokenizer.json')

In [ ]:
!tar -czvf lora_model.tar.gz lora_model

Обязательно скачайте модель локально, иначе вы утратите прогресс при сбросе колаба!

# 5. Инференс модели — 15 баллов



### Шаблон токенизации — 5 баллов

Для начала нам нужно внести шаблон диалога в наш токенизатор, чтобы его было удобно использовать. Для этого нужно модифицировать `tokenizer.chat_template`

In [ ]:
# ---- Ваш код здесь ----

# Оформляем все сообщения в том же формате, что и formatting_func.
# При генерации дополнительно открываем новую реплику ассистента.
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n'"
    " + message['content'] + '<|im_end|>\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% endif %}"
)
# Сохраняем новый шаблон вместе с токенизатором.
tokenizer.save_pretrained("lora_model")
# ---- Конец кода ----

Проверим, что мы все сделали верно:

In [42]:
chat = [
  {"role": "user", "content": "Hello!"},
  {"role": "assistant", "content": "How can I help you today?"},
  {"role": "user", "content": "I dont know!"},
]

assert tokenizer.apply_chat_template(chat, tokenize=False) == '<|im_start|>user\nHello!<|im_end|>\n<|im_start|>assistant\nHow can I help you today?<|im_end|>\n<|im_start|>user\nI dont know!<|im_end|>\n'

Теперь давайте вмержим наш LoRA адаптер в модель, чтобы хранить ее целиком. Сохраним в формате `gguf`, это понадобится нам для инференса в дальнейшем.

In [ ]:
error! model.save_pretrained_gguf("qwen25_15_ru_instruct", tokenizer,)
# Сохраненная этим путем модель показала деградацию на тесте (да сломалась совсем, чего уж), ниже сделал сохранение мимо unsloth, средствами llama.cpp

Unsloth: Merging model weights to 16-bit format...
Detected local model directory: c:\igori\pytorch\Qwen2.5-1.5B
Found HuggingFace hub cache directory: C:\Users\igori\.cache\huggingface\hub


Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:03<00:00,  3.37s/it]


Unsloth: Merge process complete. Saved to `c:\igori\pytorch\qwen25_15_ru_instruct`
Unsloth: Converting to GGUF format...
==((====))==  Unsloth: Conversion from HF to GGUF information
   \\   /|    [0] Installing llama.cpp might take 3 minutes.
O^O/ \_/ \    [1] Converting HF to GGUF bf16 might take 3 minutes.
\        /    [2] Converting GGUF bf16 to ['q8_0'] might take 10 minutes each.
 "-____-"     In total, you will have to wait at least 16 minutes.

Unsloth: llama.cpp found in the system. Skipping installation.
Unsloth: Preparing converter script...


[unsloth_zoo.llama_cpp|WARNING]Unsloth: No supported architectures (TEXT or VISION) could be determined from the original script.
[unsloth_zoo.llama_cpp|WARNING]Unsloth: Metadata branding patch target 'self.metadata = gguf.Metadata.load(...)' not found.
[unsloth_zoo.llama_cpp|WARNING]Unsloth: Qwen2MoE num_experts patch target not found.


Unsloth: [1] Converting model into bf16 GGUF format.
This might take 3 minutes...
Unsloth: Initial conversion completed! Files: ['qwen25_15_ru_instruct_gguf\\Qwen2.5-1.5B.BF16.gguf']
Unsloth: [2] Converting GGUF bf16 into q8_0. This might take 10 minutes...
Unsloth: Model files cleanup...
Unsloth: All GGUF conversions completed successfully!
Generated files: ['qwen25_15_ru_instruct_gguf\\Qwen2.5-1.5B.Q8_0.gguf']
Unsloth: No Ollama template mapping found for model './Qwen2.5-1.5B'. Skipping Ollama Modelfile
Unsloth: example usage for text only LLMs: C:\Users\igori\.unsloth\llama.cpp\build\bin\Release\llama-cli.exe --model qwen25_15_ru_instruct_gguf\Qwen2.5-1.5B.Q8_0.gguf -p "why is the sky blue?"


{'save_directory': 'qwen25_15_ru_instruct',
 'gguf_directory': 'qwen25_15_ru_instruct_gguf',
 'gguf_files': ['qwen25_15_ru_instruct_gguf\\Qwen2.5-1.5B.Q8_0.gguf'],
 'modelfile_location': None,
 'want_full_precision': False,
 'is_vlm': False,
 'fix_bos_token': False}

Если у вас заканчивается оперативная память, то на этом моменте вы можете перезагрузить среду выполнения. Главное — сохранить прогресс на локальный комптютер. В следующей ячейке пример как можно восстановить чекпоинт:

In [49]:
if False:
    from unsloth import FastLanguageModel
    import torch
    # ---- Ваш код здесь ----
    if not torch.cuda.is_available():
        raise RuntimeError("Для загрузки модели нужна CUDA-видеокарта.")
    
    # тут как в начале
    max_seq_length = 2048
    dtype = None
    load_in_4bit = True    
    # ---- Конец кода ----
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name = "lora_model",
        # ---- Ваш код здесь ----
        max_seq_length=max_seq_length,
        dtype=dtype,
        load_in_4bit=load_in_4bit,
        device_map={"": "cuda:0"},
        # ---- Конец кода ----
    )

Проверим хотя бы на 1 примере что теперь мы понимаем русский язык:

In [50]:
FastLanguageModel.for_inference(model)
inputs = tokenizer.apply_chat_template([
    {"role": "user", "content": "Почему небо синее?"},
], return_tensors = "pt", add_generation_prompt=True).to("cuda")

In [51]:
outputs = model.generate(inputs, max_new_tokens = 64, use_cache = True)
tokenizer.decode(outputs[0])

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


'<|im_start|>user\nПочему небо синее?<|im_end|>\n<|im_start|>assistant\nНебо синее, или, точнее, синее небо, происходит из-за отражения солнечного света от земли. Вот пошаговое объяснение:\n\n1. **Солнечный свет** - это свет, который мы видим,'

Теперь снова замерим нашу модель на бенчмарках, чтобы убедиться, что мы ничего не ухудшили:

In [52]:
%%writefile run_lmeh_after.sh
#!/usr/bin/env bash
# ---- Ваш код здесь ----
set -euo pipefail

PYTHON="${PYTHON:-python}"
export PYTHONIOENCODING=utf-8
export PYTHONUNBUFFERED=1
export TQDM_MININTERVAL=5

# Загружаем прежнюю базовую модель и подключаем обученный адаптер.
# Остальные параметры оценки совпадают с исходным замером.
"$PYTHON" -u -m lm_eval run \
    --model hf \
    --model_args "pretrained=./Qwen2.5-1.5B,peft=./lora_model,dtype=float16,load_in_4bit=True,bnb_4bit_quant_type=nf4,bnb_4bit_compute_dtype=float16,bnb_4bit_use_double_quant=True" \
    --tasks truthfulqa_ru_mc1,copa,wsc273 \
    --num_fewshot 0 \
    --device cuda:0 \
    --batch_size 8 \
    --seed 42 \
    --output_path ./results/after_training
# ---- Конец кода ----

Writing run_lmeh_after.sh


In [53]:
# ---- Ваш код здесь ----
import os
import sys
import subprocess
from pathlib import Path

script_path = Path("run_lmeh_after.sh")
log_path = Path("run_lmeh_after.log")

# Заменяем переносы строк Windows на Unix для Bash.
script_path.write_bytes(
    script_path.read_bytes().replace(b"\r\n", b"\n")
)

env = os.environ.copy()

# Запускаем Python из той же среды, что и ядро ноутбука.
env["PYTHON"] = Path(sys.executable).as_posix()
env["PYTHONIOENCODING"] = "utf-8"
env["PYTHONUNBUFFERED"] = "1"

# Ограничиваем частоту обновления индикаторов прогресса.
env["TQDM_MININTERVAL"] = "5"

with log_path.open("w", encoding="utf-8") as log_file:
    with subprocess.Popen(
        # Используем Git Bash, а не оболочку WSL.
        ["C:/Program Files/Git/bin/bash.exe", str(script_path)],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,  # Объединяем обычный вывод и сообщения ошибок.
        text=True,
        encoding="utf-8",
        errors="replace",
        bufsize=1,
    ) as process:
        for line in process.stdout:
            # Показываем ход выполнения и одновременно сохраняем полный лог.
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()

        return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Harness завершился с кодом {return_code}. "
        f"Подробности — в выводе выше и в {log_path}."
    )

print("\nБенчмарки завершены. Результаты: results/after_training")
# ---- Конец кода ----

C:\igori\pytorch\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
2026-10-16:10:47:23 INFO     [_cli.run:413] Selected Tasks: ['truthfulqa_ru_mc1', 'copa', 'wsc273']
2026-10-16:10:47:23 INFO     [evaluator:214] Setting random seed to 42 | Setting numpy seed to 42 | Setting torch manual seed to 42 | Setting fewshot manual seed to 42
2026-10-16:10:47:23 INFO     [evaluator:239] Initializing hf model, with arguments: {'pretrained': './Qwen2.5-1.5B', 'peft': './lora_model', 'dtype': 'float16', 'load_in_4bit': True, 'bnb_4bit_quant_type': 'nf4', 'bnb_4bit_compute_dtype': 'float16', 'bnb_4bit_use_double_quant': True}
2026-10-16:10:47:24 INFO     [models.huggingface:289] Using device 'cuda:0'
2026-10-16:10:47:25 INFO     [models.huggingface:582] Model parallel was set to False, max memory was not set, and device map was set to {'': 'cuda:0'}
W1016

Стало после обучения

| Бенчмарк | Accuracy | Стандартная ошибка |
|-----------------|-----:|-----:|
|truthfulqa_ru_mc1| 30.20% | 1.64% |
|copa             | 83% | 3.78% |
|wsc273           | 75.82% | 2.60 % |

Было:

| Бенчмарк | Accuracy | Стандартная ошибка |
|:---|---:|---:|
| truthfulqa_ru_mc1 | 28.93% | 1.62% |
| copa | 81% | 3.94% |
| wsc273 | 76.19% | 2.58% |

Видим, что показатели в целом не ухудшились, но и улучшением это назвать сложно

### Инференс llama.cpp — 5 баллов
Теперь давайте запустим инференс нашей модели через `llama.cpp`. Это один из самых популярных фреймворков для инференса языковых моделей локально. Его документацию можно почитать [тут](https://github.com/ggml-org/llama.cpp/tree/master?tab=readme-ov-file#quick-start). Запустить его нужно в режиме сервера через терминал в Google Colab (внизу в футтере). А из ноутбука с помощью OpenAI API прокачать:

In [ ]:
error Уже раньше unsloth собрал при экспорте gguf
!git clone https://github.com/ggml-org/llama.cpp
!cd llama.cpp && cmake -B build && cmake --build build --config Release

cmake -S "~\.unsloth\llama.cpp" -B "~\.unsloth\llama.cpp\build-cuda-vs2019" -G "Visual Studio 16 2019" -A x64 -T "cuda=C:\Program Files\NVIDIA GPU Computing Toolkit\CUDA\v13.0" -DGGML_CUDA=ON
cmake --build "~\.unsloth\llama.cpp\build-cuda-vs2019" --config Release --target llama-server -j 8



In [ ]:
# Сольём воедино адаптер с моделью, только заново и аккуратно

import os
import sys
import subprocess

merge_code = r'''
from pathlib import Path
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel

base_path = "./Qwen2.5-1.5B"
adapter_path = "./lora_model"
output_path = Path("qwen25_15_ru_merged_peft")

# Защищаем результат предыдущего запуска от перезаписи.
if output_path.exists():
    raise FileExistsError(
        f"{output_path} уже существует. Выберите другое имя каталога."
    )

print("Загружаем исходную модель на CPU в float32...", flush=True)
model = AutoModelForCausalLM.from_pretrained(
    base_path,
    torch_dtype=torch.float32,
    device_map={"": "cpu"},
    low_cpu_mem_usage=True,
)

print("Подключаем сохранённый LoRA-адаптер...", flush=True)
model = PeftModel.from_pretrained(
    model,
    adapter_path,
    is_trainable=False,
)
model.eval()
print("Активные адаптеры:", model.active_adapters, flush=True)

print("Объединяем веса стандартным PEFT...", flush=True)

# Добавляем поправки LoRA к базовым весам и удаляем обёртку адаптера.
# safe_merge проверяет наличие нечисловых значений при слиянии.
merged_model = model.merge_and_unload(safe_merge=True)

print("Сохраняем объединённую модель в float32...", flush=True)
merged_model.save_pretrained(
    output_path,
    safe_serialization=True,
    max_shard_size="2GB",
)

# Загружаем токенизатор адаптера и явно задаём обучающий шаблон:
# в lora_model он мог быть сохранён до изменения chat_template.
tokenizer = AutoTokenizer.from_pretrained(adapter_path)
tokenizer.chat_template = (
    "{% for message in messages %}"
    "{{ '<|im_start|>' + message['role'] + '\n'"
    " + message['content'] + '<|im_end|>\n' }}"
    "{% endfor %}"
    "{% if add_generation_prompt %}"
    "{{ '<|im_start|>assistant\n' }}"
    "{% endif %}"
)
tokenizer.save_pretrained(output_path)

print(f"Готово: {output_path.resolve()}", flush=True)
'''

env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"

with open("merge_peft.log", "w", encoding="utf-8") as log_file:
    with subprocess.Popen(
        [sys.executable, "-u", "-c", merge_code],
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()

        return_code = process.wait()

if return_code != 0:
    raise RuntimeError(
        f"Слияние завершилось с кодом {return_code}. "
        "Подробности выше и в merge_peft.log."
    )


c:\igori\pytorch\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
W1016 11:32:34.452000 18552 Lib\site-packages\torch\distributed\elastic\multiprocessing\redirects.py:29] NOTE: Redirects are currently not supported in Windows or MacOs.
Загружаем исходную модель на CPU в float32...
`torch_dtype` is deprecated! Use `dtype` instead!
Подключаем сохранённый LoRA-адаптер...
Активные адаптеры: ['default']
Объединяем веса стандартным PEFT...
Сохраняем объединённую модель в float32...
Готово: C:\igori\pytorch\qwen25_15_ru_merged_peft


In [ ]:
# Упакуем в GGUF и квантуем в Q8

import os
import sys
import subprocess
from pathlib import Path

llama_path = Path.home() / ".unsloth" / "llama.cpp"
model_path = Path("qwen25_15_ru_merged_peft").resolve()
output_path = Path("qwen25_15_ru_merged_peft_gguf").resolve()

converter = llama_path / "convert_hf_to_gguf.py"
quantizer = llama_path / "build" / "bin" / "Release" / "llama-quantize.exe"

for path in (model_path, converter, quantizer):
    if not path.exists():
        raise FileNotFoundError(f"Не найдено: {path}")

# Не перезаписываем результаты предыдущей конвертации.
output_path.mkdir(exist_ok=False)

bf16_path = output_path / "qwen-merged.BF16.gguf"
q8_path = output_path / "qwen-merged.Q8_0.gguf"

env = os.environ.copy()
env["PYTHONIOENCODING"] = "utf-8"

# Конвертер использует gguf-py из собственного репозитория.
env.pop("NO_LOCAL_GGUF", None)


def run_conversion(command, log_file):
    """Показываем сообщения процесса и сохраняем их в лог."""
    with subprocess.Popen(
        command,
        env=env,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        encoding="utf-8",
        errors="replace",
    ) as process:
        for line in process.stdout:
            print(line, end="", flush=True)
            log_file.write(line)
            log_file.flush()

        return_code = process.wait()

    if return_code != 0:
        raise RuntimeError(
            f"Команда завершилась с кодом {return_code}. "
            "Подробности выше и в convert_peft_gguf.log."
        )


with open("convert_peft_gguf.log", "w", encoding="utf-8") as log_file:
    # Сначала меняем формат хранения и приводим веса к BF16.
    run_conversion(
        [
            sys.executable, "-u", str(converter),
            str(model_path),
            "--outfile", str(bf16_path),
            "--outtype", "bf16",
        ],
        log_file,
    )

    # Затем квантуем BF16 в Q8_0.
    # BF16-файл сохраняем для отдельной проверки без Q8-квантования.
    run_conversion(
        [
            str(quantizer),
            str(bf16_path),
            str(q8_path),
            "Q8_0",
        ],
        log_file,
    )

print(f"\nBF16: {bf16_path}")
print(f"Q8_0: {q8_path}")


c:\igori\pytorch\.venv\Lib\site-packages\requests\__init__.py:113: RequestsDependencyWarning: urllib3 (2.5.0) or chardet (6.0.0.post1)/charset_normalizer (3.4.4) doesn't match a supported version!
  warnings.warn(
INFO:hf-to-gguf:Loading model: qwen25_15_ru_merged_peft
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: loading model weight map from 'model.safetensors.index.json'
INFO:hf-to-gguf:gguf: indexing model part 'model-00001-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00002-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00003-of-00004.safetensors'
INFO:hf-to-gguf:gguf: indexing model part 'model-00004-of-00004.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float32 --> BF16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float32 --> F32, shape = {1536}
INFO:hf-to-

In [ ]:
# Теперь запускаем сервер руками

# !~\.unsloth\llama.cpp\build-cuda-vs2019\bin\Release\llama-server.exe --model ".\qwen25_15_ru_merged_peft_gguf\qwen-merged.Q8_0.gguf" --alias qwen-local --host 127.0.0.1 --port 8080 --n-gpu-layers 99 --ctx-size 2048 --parallel 1 --jinja

# и начинаем с ним работать

import openai

client = openai.Client(
    # ---- Ваш код здесь ----
    base_url="http://127.0.0.1:8080/v1",
    api_key="local",  # Формальное значение для клиента; сервер ключ не требует.
    timeout=120.0,    
    # ---- Конец кода ----
)

response = client.chat.completions.create(
    # ---- Ваш код здесь ----
    model="qwen-local",
    messages=[
        {"role": "user", "content": "Привет! Скажи, почему небо синего цвета?"},
    ],
    temperature=0,  # Жадный выбор токенов для воспроизводимого сравнения.
    max_tokens=256,    
    # ---- Конец кода ----
)

In [66]:
response.choices[0].message.content

'Привет! Небо синего цвета — это не просто красивая картинка, а результат сложного физического процесса, который происходит в небесах. Вот основные причины, почему небо кажется синим:\n\n1. **Солнечный свет**: Небо синее, в основном, потому, что солнечный свет, который падает на небо, преобразуется в синий цвет. Солнечный свет состоит из всех цветов радуги, но именно синий цвет преобразуется в синий, когда он падает на небо.\n\n2. **Отражение солнечного света**: Солнечный свет, падающий на небо, отражается от земли и других объектов на небе. Это отражение создает эффект, когда солнечный свет, падающий на небо, кажется синим.\n\n3. **Отражение от земли**: Земля, которая находится на поверхности Земли, также отражает солнечный свет. Это отражение отражает солн'

### Прогон на корзинке — 5 баллов

Прогоним корзинку из самого начала и убедимся что наша модель стала хорошо отвечать на диалоги:

In [67]:
# ---- Ваш код здесь ----
import json
from pathlib import Path

baseline = json.loads(
    Path("baseline_two_modes.json").read_text(encoding="utf-8")
)

result_path = Path("after_training_llamacpp_two_modes.json")
if result_path.exists():
    raise FileExistsError(
        f"{result_path} уже существует. Выбери другое имя для нового прогона."
    )

after_answers = []

for item in baseline["results"]:
    result = {
        "number": item["number"],
        "messages": item["messages"],
    }

    print(f"\n{'=' * 60}\nЗадание {item['number']}")

    for mode, label in [
        ("without_chat_template", "Без шаблона"),
        ("with_chat_template", "С прежним шаблоном Qwen"),
    ]:
        # Берём именно тот текст, который подавали базовой модели.
        input_text = item[mode]["input_text"]

        completion = client.completions.create(
            model="qwen-local",
            prompt=input_text,
            temperature=0,
            max_tokens=baseline["generation"]["max_new_tokens"],
            # Отключаем штраф повторений для близости к прежней генерации.
            extra_body={"repeat_penalty": 1.0},
        )
        answer = completion.choices[0].text.strip()

        result[mode] = {
            "input_text": input_text,
            "answer": answer,
            "score": None,
        }

        print(f"\n{label}")
        print(f"До обучения:\n{item[mode]['answer']}")
        print(f"После обучения:\n{answer}")

    after_answers.append(result)

after_data = {
    "stage": "after_training",
    "backend": "llama.cpp",
    "model": "qwen-local",
    "baseline_file": "baseline_two_modes.json",
    "generation": {
        "temperature": 0,
        "max_tokens": baseline["generation"]["max_new_tokens"],
        "repeat_penalty": 1.0,
    },
    "results": after_answers,
}

result_path.write_text(
    json.dumps(after_data, ensure_ascii=False, indent=2),
    encoding="utf-8",
)
print(f"\nРезультаты сохранены: {result_path.resolve()}")
# ---- Конец кода ----


Задание 1

Без шаблона
До обучения:
Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на вопрос, который я задал себе. Я не могу найти ответ на воп

## **Выводы**

До обучения почти все ответы на вопросы на русском языке были бессвязными, а при добавлении шаблона - совершенно мусорными. Хотя вопрос на китайском языке получил хороший по форме и содержанию ответ.

После обучения, из десяти тестов в шести ответ был корректным по форме и достаточно осмысленным (что удивительно для модели 1.5B), в оставшихся четырех ответ был хорошим, но генерация не остановилась по окончании ответа и начался повтор с посторонними токенами. Возможно, это также можно исправить дообучением - в документации TRL описано обучение только на ответах с маркером конца ответа. Добавление шаблона диалога не ухудшает генерацию, форма ответа даже улучшается.

Модель после обучения стала отвечать более корректно, соблюдая формат диалога. Бенчмарки не ухудшились, но и уверенного роста не показали.

Обучение на 8000 примерах заняло 500 шагов и 7 часов на RTX 5080. После первоначального снижения training loss колебался около 1.2 с небольшой тенденцией к уменьшению. Количество шагов задано на глаз, на будущее можно сделать отдельную валидационную выборку в 200 диалогов и проверять лосс на ней.

Причина долгого обучения в неудачном выборе параметров - выход за границы доступной видеопамяти резко уменьшил скорость. Надо уменьшить размер батча с 4 до 2 - количество микробатчей возрастает в два раза, но скорость одного шага в тесте повысилась в шесть раз.

Объединение и экспорт через Unsloth были неудачными, причину не искал. Слияние через PEFT и конвертация средствами llama.cpp сохранили качество обученной модели.

Размер LoRA-адаптера всего 70 МБ на модели в 1.5B. Адаптер добавлен на всех матрицах слоёв (кроме нормализации, эмбеддингов, и выходной проекции), базовые веса заморожены.

В целом, научился делать адаптацию модели при наличии хороших данных для обучения. Осознал ресурсоемкость этого процесса.